# Evaluation on Large Images with Patchification (Dual Channel)<br>
<br>
This notebook demonstrates how to:<br>
1. Load two trained models (Channel 1 and Channel 2)<br>
2. Patch large images into smaller tiles<br>
3. Run predictions on patches using both models simultaneously<br>
4. Reconstruct full-size predictions for both channels<br>
5. Combine original 4 channels + 2 predictions = 6 channels<br>
6. Visualize and save results

## Setup and Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('..')

In [24]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from pathlib import Path
import json
import tifffile as tf
from tqdm.notebook import tqdm

In [4]:
from utils.patchify import (
    predict_large_image,
    patchify_image,
    unpatchify_image
)
from models.model_factory import ModelFactory
from cilia_utils.utils import *

## Configuration

Model configuration

In [76]:
MODEL_CHECKPOINT_CH1 = '../outputs_07Jan25_12-34-00/c1_unet_tiny/best_model.pth'
MODEL_CHECKPOINT_CH2 = '../outputs_07Jan25_12-34-00/c2_unet_tiny/best_model.pth'
ARCHITECTURE = 'unet_tiny'

Channel indices

In [77]:
C1_IDX = 2
C2_IDX = 3

Inference configuration

In [101]:
PATCH_SIZE = 64  # Must match training image size
OVERLAP = 48      # Overlap between patches
BATCH_SIZE = 512
THRESHOLD = 0.5
BLEND_MODE = 'average'  # 'average', 'max', or 'first'

Device

In [88]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

Using device: cuda


Input/Output paths

In [89]:
ROOT = Path(
    '/group/jug/aman/Cilia_Datasets/extracted/'
    'W19 - 2025_Pkhd1_cells/'
    'W19 - 2025_Pkhd1 cells'
)
PRED_ROOT = Path("/group/jug/aman/Cilia_Datasets/Predictions")
# PRED_ROOT = Path("/group/jug/aman/Cilia_Datasets/Predictions_TEMP")

## 1. Load Both Models

In [90]:
def load_model(checkpoint_path, architecture, device='cuda'):
    """Load trained model from checkpoint."""
    model = ModelFactory.create_model(
        architecture,
        num_classes=1,
        pretrained=False
    )
    
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    
    print(f"Loaded model from: {checkpoint_path}")
    print(f"Epoch: {checkpoint.get('epoch', 'N/A')}")
    print(f"Metrics: {checkpoint.get('metrics', 'N/A')}")
    
    return model

Load both models

In [91]:
model_ch1 = load_model(MODEL_CHECKPOINT_CH1, ARCHITECTURE, DEVICE)
model_ch2 = load_model(MODEL_CHECKPOINT_CH2, ARCHITECTURE, DEVICE)

Loaded model from: ../outputs_07Jan25_12-34-00/c1_unet_tiny/best_model.pth
Epoch: 73
Metrics: {'loss': 0.17193135246634483, 'iou': 0.755072608590126}
Loaded model from: ../outputs_07Jan25_12-34-00/c2_unet_tiny/best_model.pth
Epoch: 90
Metrics: {'loss': 0.13879333157092333, 'iou': 0.7455189079046249}


In [92]:
print("\n✓ Both models loaded successfully")


✓ Both models loaded successfully


## 2. Load All Test Images

In [93]:
def load_all_images_from_root(root_path):
    """
    Recursively load all TIFF images from directory structure.
    
    Returns:
        all_slices: (N_total, C, H, W) stacked 2D slices
        slice_index: List of metadata dicts for each slice
    """
    all_slices = []
    slice_index = []
    
    print("Scanning folders for TIFF files...")
    
    for cropped_dir in root_path.rglob('Cropped original image'):
        if not cropped_dir.is_dir():
            continue
        
        print(f"Found: {cropped_dir}")
        
        for tif_path in cropped_dir.glob('*.tif'):
            img = tf.imread(tif_path)
            
            # Normalize to (Z, C, Y, X)
            if img.ndim == 3:  # (C, Y, X)
                img = img[np.newaxis, ...]
            elif img.ndim != 4:
                raise ValueError(f"Unexpected shape {img.shape} in {tif_path}")
            
            Z, C, Y, X = img.shape
            
            for z in range(Z):
                all_slices.append(img[z])  # (C, Y, X)
                slice_index.append({
                    "path": tif_path,
                    "z": z,
                    "Z": Z
                })
    
    all_slices = np.stack(all_slices, axis=0)  # (N_total, C, Y, X)
    print(f"✓ Total 2D slices collected: {all_slices.shape[0]}")
    
    return all_slices, slice_index

Load images

In [94]:
all_slices, slice_index = load_all_images_from_root(ROOT)
print("Slices:", len(all_slices))
print("Slice index entries:", len(slice_index))

from collections import Counter
counts = Counter([info["path"] for info in slice_index])
print("Per-file Z counts:")
for k, v in counts.items():
    print(k.name, "->", v)


Scanning folders for TIFF files...
Found: /group/jug/aman/Cilia_Datasets/extracted/W19 - 2025_Pkhd1_cells/W19 - 2025_Pkhd1 cells/CCDC92, Y-tub, Arl13b, DAPI/Pkhd1 KO/Cropped original image
Found: /group/jug/aman/Cilia_Datasets/extracted/W19 - 2025_Pkhd1_cells/W19 - 2025_Pkhd1 cells/CCDC92, Y-tub, Arl13b, DAPI/Pkhd1 CTRL/Cropped original image
Found: /group/jug/aman/Cilia_Datasets/extracted/W19 - 2025_Pkhd1_cells/W19 - 2025_Pkhd1 cells/ZDHHC5, Y-tub, Arl13, DAPI/Pkhd1 KO/Cropped original image
Found: /group/jug/aman/Cilia_Datasets/extracted/W19 - 2025_Pkhd1_cells/W19 - 2025_Pkhd1 cells/ZDHHC5, Y-tub, Arl13, DAPI/Pkhd1 CTRL/Cropped original image
✓ Total 2D slices collected: 510
Slices: 510
Slice index entries: 510
Per-file Z counts:
IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_3_original cropped.tif -> 15
IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_4_original cropped.tif -> 19
IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_7_original cropped.tif -> 15
IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, 

In [95]:
all_slices.shape

(510, 4, 675, 675)

## 3. Prepare Images for Inference

In [96]:
def prepare_image_for_single_channel(image, channel_idx):
    """
    Prepare single-channel image for model inference (convert to 3-channel RGB).
    
    Args:
        image: (C, H, W) multichannel image
        channel_idx: Which channel to extract and use as RGB
    
    Returns:
        RGB image (3, H, W) ready for model
    """
    channel = image[channel_idx]
    channel_norm = normalize_channel(channel)
    rgb = np.stack([channel_norm] * 3, axis=0)
    
    # Keep pixel values in [0, 255] range to match training
    return (rgb * 255).astype(np.uint8).astype(np.float32)

Test on first slice

In [97]:
test_idx = 0
raw_image = all_slices[test_idx]  # (C, H, W)
prepared_ch1 = prepare_image_for_single_channel(raw_image, C1_IDX)
prepared_ch2 = prepare_image_for_single_channel(raw_image, C2_IDX)

In [98]:
print(f"Raw image shape: {raw_image.shape}")
print(f"Prepared Ch1 shape: {prepared_ch1.shape}")
print(f"Prepared Ch2 shape: {prepared_ch2.shape}")

Raw image shape: (4, 675, 675)
Prepared Ch1 shape: (3, 675, 675)
Prepared Ch2 shape: (3, 675, 675)


## 4. Batch Inference on All Slices

In [99]:
from concurrent.futures import ThreadPoolExecutor

def predict_dual_channel(model_ch1, model_ch2, image, patch_size, overlap, 
                         batch_size, blend_mode, device, threshold, c1_idx, c2_idx):
    """
    Run inference on both channels independently using ThreadPoolExecutor.
    
    Args:
        model_ch1, model_ch2: Trained models for each channel
        image: (C, H, W) raw image
        c1_idx, c2_idx: Channel indices
        patch_size: Patch size for inference
        overlap: Overlap between patches
        batch_size: Batch size for inference
        blend_mode: Blending mode for patch reconstruction
        device: Device to run inference on
        threshold: Threshold for binary prediction
    
    Returns:
        pred_mask_ch1: (H, W) channel 1 prediction mask
        pred_binary_ch1: (H, W) channel 1 binary prediction
        pred_mask_ch2: (H, W) channel 2 prediction mask
        pred_binary_ch2: (H, W) channel 2 binary prediction
    """
    # Prepare inputs for both channels
    prepared_ch1 = prepare_image_for_single_channel(image, c1_idx)
    prepared_ch2 = prepare_image_for_single_channel(image, c2_idx)
    
    def infer_channel(model, prepared_image):
        """Helper function to run inference on a single channel."""
        return predict_large_image(
            model=model,
            image=prepared_image,
            patch_size=patch_size,
            overlap=overlap,
            batch_size=batch_size,
            blend_mode=blend_mode,
            device=device,
            threshold=threshold
        )
    
    # Run both inferences in parallel
    with ThreadPoolExecutor(max_workers=2) as executor:
        future_ch1 = executor.submit(infer_channel, model_ch1, prepared_ch1)
        future_ch2 = executor.submit(infer_channel, model_ch2, prepared_ch2)
        
        # Get results (blocks until complete)
        pred_mask_ch1, pred_binary_ch1 = future_ch1.result()
        pred_mask_ch2, pred_binary_ch2 = future_ch2.result()
    
    return pred_mask_ch1, pred_binary_ch1, pred_mask_ch2, pred_binary_ch2


Run inference on all slices

In [102]:
from tqdm.notebook import tqdm 
all_pred_masks_ch1 = []
all_pred_masks_ch2 = []

all_pred_binary_ch1 = []
all_pred_binary_ch2 = []

assert len(all_pred_masks_ch1) == 0
assert len(all_pred_masks_ch2) == 0

print("Running inference on all slices...")
for i, raw_slice in tqdm(enumerate(all_slices), total=len(all_slices), desc="Inference", leave = False):
# for i, raw_slice in tqdm(enumerate(range(15)), desc="Inference", leave = False):
    pred_mask_ch1, pred_binary_ch1, pred_mask_ch2, pred_binary_ch2 = predict_dual_channel(
        model_ch1=model_ch1,
        model_ch2=model_ch2,
        image=raw_slice,
        patch_size=PATCH_SIZE,
        overlap=OVERLAP,
        batch_size=BATCH_SIZE,
        blend_mode=BLEND_MODE,
        device=DEVICE,
        threshold=THRESHOLD,
        c1_idx=C1_IDX,
        c2_idx=C2_IDX
    )
    
    all_pred_masks_ch1.append(pred_mask_ch1)
    all_pred_binary_ch1.append(pred_binary_ch1)
    all_pred_masks_ch2.append(pred_mask_ch2)
    all_pred_binary_ch2.append(pred_binary_ch2)

Running inference on all slices...


Inference:   0%|          | 0/510 [00:00<?, ?it/s]

Predicting patches:   0%|          | 0/4 [00:00<?, ?it/s]

Predicting patches:   0%|          | 0/4 [00:00<?, ?it/s]

Predicting patches:   0%|          | 0/4 [00:00<?, ?it/s]

...

Predicting patches:   0%|          | 0/4 [00:00<?, ?it/s]

Predicting patches:   0%|          | 0/4 [00:00<?, ?it/s]

Predicting patches:   0%|          | 0/4 [00:00<?, ?it/s]

Predicting patches:   0%|          | 0/4 [00:00<?, ?it/s]

Predicting patches:   0%|          | 0/4 [00:00<?, ?it/s]

Stack into arrays

In [103]:
all_pred_masks_ch1 = np.stack(all_pred_masks_ch1, axis=0)  # (N_total, H, W)
all_pred_binary_ch1 = np.stack(all_pred_binary_ch1, axis=0)
all_pred_masks_ch2 = np.stack(all_pred_masks_ch2, axis=0)
all_pred_binary_ch2 = np.stack(all_pred_binary_ch2, axis=0)

In [104]:
print("✓ All slices processed")
print(f"  Channel 1 mask shape: {all_pred_masks_ch1.shape}")
print(f"  Channel 2 mask shape: {all_pred_masks_ch2.shape}")

✓ All slices processed
  Channel 1 mask shape: (510, 675, 675)
  Channel 2 mask shape: (510, 675, 675)


## 5. Organize Predictions by File

In [105]:
def organize_predictions_by_file(slice_index, predictions_ch1, predictions_ch2):
    """
    Reorganize flat prediction arrays into per-file dictionaries.
    
    Returns:
        pred_dict_ch1, pred_dict_ch2: dicts mapping file paths to (Z, H, W) arrays
    """
    pred_dict_ch1 = {}
    pred_dict_ch2 = {}
    
    for info, pred_ch1, pred_ch2 in zip(slice_index, predictions_ch1, predictions_ch2):
        path = info["path"]
        z = info["z"]
        Z = info["Z"]
        
        if path not in pred_dict_ch1:
            pred_dict_ch1[path] = [None] * Z
            pred_dict_ch2[path] = [None] * Z
        
        pred_dict_ch1[path][z] = pred_ch1
        pred_dict_ch2[path][z] = pred_ch2
    
    # Stack Z slices for each file
    for path in pred_dict_ch1:
        pred_dict_ch1[path] = np.stack(pred_dict_ch1[path], axis=0)
        pred_dict_ch2[path] = np.stack(pred_dict_ch2[path], axis=0)
    
    return pred_dict_ch1, pred_dict_ch2

In [106]:
pred_dict_ch1, pred_dict_ch2 = organize_predictions_by_file(
    slice_index, 
    all_pred_binary_ch1, 
    all_pred_binary_ch2
)

In [107]:
print("Organized predictions by file:")
for path in pred_dict_ch1:
    print(f"  {path.name}: Ch1={pred_dict_ch1[path].shape}, Ch2={pred_dict_ch2[path].shape}")

Organized predictions by file:
  IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_3_original cropped.tif: Ch1=(15, 675, 675), Ch2=(15, 675, 675)
  IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_4_original cropped.tif: Ch1=(19, 675, 675), Ch2=(19, 675, 675)
  IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_7_original cropped.tif: Ch1=(15, 675, 675), Ch2=(15, 675, 675)
  IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_5_original cropped.tif: Ch1=(16, 675, 675), Ch2=(16, 675, 675)
  IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_1_original cropped.tif: Ch1=(16, 675, 675), Ch2=(16, 675, 675)
  IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_8_original cropped.tif: Ch1=(17, 675, 675), Ch2=(17, 675, 675)
  IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_6_original cropped.tif: Ch1=(16, 675, 675), Ch2=(16, 675, 675)
  IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_2_original cropped.tif: Ch1=(18, 675, 675), Ch2=(18, 675, 675)
  IMCD3 Pkhd CTRL - CCDC92, ARL13B, Y-TUB, DAPI_2_original cropped.tif: Ch1=(15, 675, 675), Ch2=(

## 6. Create 6-Channel Output and Save

In [108]:
def load_original_image_full(tif_path):
    """
    Load full original image without slicing.
    
    Returns:
        img: (Z, C, Y, X) array
    """
    img = tf.imread(tif_path)
    
    if img.ndim == 3:  # (C, Y, X)
        img = img[np.newaxis, ...]
    elif img.ndim != 4:
        raise ValueError(f"Unexpected shape {img.shape}")
    
    return img

In [109]:
def create_6channel_output(original_img, pred_dict_ch1, pred_dict_ch2, tif_path):
    """
    Combine original 4 channels + 2 prediction channels = 6 channels
    with order: [0, 1, pred_ch1, 2, pred_ch2, 3]

    Args:
        original_img: (Z, 4, Y, X)
        pred_dict_ch1: dict[path -> (Z, Y, X)]
        pred_dict_ch2: dict[path -> (Z, Y, X)]
        tif_path: Path key

    Returns:
        output: (Z, 6, Y, X)
    """
    Z, C, Y, X = original_img.shape
    assert C == 4, f"Expected 4 original channels, got {C}"

    pred_ch1 = pred_dict_ch1[tif_path]
    pred_ch2 = pred_dict_ch2[tif_path]

    assert pred_ch1.shape == (Z, Y, X)
    assert pred_ch2.shape == (Z, Y, X)

    # output = np.zeros((Z, 6, Y, X), dtype=np.float32)
    output = np.zeros((Z, 6, Y, X), dtype=np.uint16)

    # 0, 1
    output[:, 0, :, :] = original_img[:, 0, :, :]
    output[:, 1, :, :] = original_img[:, 1, :, :]

    # pred_ch1
    output[:, 2, :, :] = (pred_ch1.astype(np.uint16) * 5961)
    # output[:, 2, :, :] = pred_ch1#.astype(np.uint16) 

    # 2
    output[:, 3, :, :] = original_img[:, 2, :, :]

    # pred_ch2
    output[:, 4, :, :] = (pred_ch2.astype(np.uint16) * 5961)
    # output[:, 4, :, :] = pred_ch2#.astype(np.uint16)

    # 3
    output[:, 5, :, :] = original_img[:, 3, :, :]
    
    return output


Create output directory

In [110]:
PRED_ROOT.mkdir(parents=True, exist_ok=True)

In [111]:
print(f"Saving 6-channel outputs to: {PRED_ROOT}")
for tif_path in tqdm(pred_dict_ch1):
# for tif_path in tqdm(range(5)):
    # Load original image
    original_img = load_original_image_full(tif_path)
    
    # Create 6-channel output
    output_6channel = create_6channel_output(
        original_img, 
        pred_dict_ch1, 
        pred_dict_ch2, 
        tif_path
    )
    print(output_6channel.shape)
    # Get relative path for mirrored directory structure
    rel_path = tif_path.relative_to(ROOT)
    rel_dir = rel_path.parent
    
    # Create output directory
    out_dir = PRED_ROOT / rel_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    
    # Save 6-channel TIFF
    output_filename = tif_path.name.replace(".tif", "_6channel.tif")
    output_path = out_dir / output_filename

    tf.imwrite(
        output_path,
        # output_6channel.astype(np.float32),
        output_6channel.astype(np.uint16),
        imagej=True,
        metadata={'axes':'ZCYX'}
    )
        
    print(f"✓ Saved {output_filename} ({output_6channel.shape}) to {out_dir}")
    # break

Saving 6-channel outputs to: /group/jug/aman/Cilia_Datasets/Predictions


  0%|          | 0/34 [00:00<?, ?it/s]

(15, 6, 675, 675)
✓ Saved IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_3_original cropped_6channel.tif ((15, 6, 675, 675)) to /group/jug/aman/Cilia_Datasets/Predictions/CCDC92, Y-tub, Arl13b, DAPI/Pkhd1 KO/Cropped original image
(19, 6, 675, 675)
✓ Saved IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_4_original cropped_6channel.tif ((19, 6, 675, 675)) to /group/jug/aman/Cilia_Datasets/Predictions/CCDC92, Y-tub, Arl13b, DAPI/Pkhd1 KO/Cropped original image
(15, 6, 675, 675)
✓ Saved IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_7_original cropped_6channel.tif ((15, 6, 675, 675)) to /group/jug/aman/Cilia_Datasets/Predictions/CCDC92, Y-tub, Arl13b, DAPI/Pkhd1 KO/Cropped original image
(16, 6, 675, 675)
✓ Saved IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_5_original cropped_6channel.tif ((16, 6, 675, 675)) to /group/jug/aman/Cilia_Datasets/Predictions/CCDC92, Y-tub, Arl13b, DAPI/Pkhd1 KO/Cropped original image
(16, 6, 675, 675)
✓ Saved IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_1_original cropped

In [112]:
print("✓ All 6-channel outputs saved")

✓ All 6-channel outputs saved


## 7. Visualization Helper

In [85]:
from matplotlib import patches
def visualize_dual_predictions(
    original_img, pred_ch1, pred_ch2,
    z_slice=None, figsize=(20, 16),
    patch_size=128, seed=None, show_patch_box=True,
    transpose=False  # NEW
):
    """
    Visualize original channels, predictions, and a random zoomed patch.
    
    Args:
        transpose: if True, swaps last two axes (X/Y) for display
    """

    if seed is not None:
        np.random.seed(seed)

    # Handle 4D input
    if original_img.ndim == 4:
        if z_slice is None:
            z_slice = 0
        original_img = original_img[z_slice]
        pred_ch1 = pred_ch1[z_slice] if pred_ch1.ndim == 3 else pred_ch1
        pred_ch2 = pred_ch2[z_slice] if pred_ch2.ndim == 3 else pred_ch2

    # Transpose if needed
    if transpose:
        original_img = original_img[..., ::-1].copy() if original_img.ndim == 3 else original_img.transpose(0,2,1)
        pred_ch1 = pred_ch1.T
        pred_ch2 = pred_ch2.T

    H, W = pred_ch1.shape

    # --------------------
    # Random patch
    # --------------------
    ph = pw = patch_size
    y0 = np.random.randint(0, H - ph)
    x0 = np.random.randint(0, W - pw)

    # --------------------
    # Normalize originals
    # --------------------
    img_ch1 = original_img[C1_IDX]
    img_ch2 = original_img[C2_IDX]

    img_ch1_norm = (img_ch1 - img_ch1.min()) / (img_ch1.max() - img_ch1.min() + 1e-8)
    img_ch2_norm = (img_ch2 - img_ch2.min()) / (img_ch2.max() - img_ch2.min() + 1e-8)

    overlay = np.stack([img_ch1_norm, img_ch2_norm, np.zeros_like(img_ch1_norm)], axis=2)
    combined_pred = np.stack([pred_ch1, pred_ch2, np.zeros_like(pred_ch1)], axis=2)

    # --------------------
    # Figure
    # --------------------
    fig, axes = plt.subplots(3, 3, figsize=figsize)
    fig.suptitle(f'Dual Channel Predictions (Z={z_slice})', fontsize=16)

    # Row 1: full images
    axes[0, 0].imshow(img_ch1_norm, cmap='viridis')
    axes[0, 0].set_title(f'Original Channel {C1_IDX}')
    axes[0, 0].axis('off')

    axes[0, 1].imshow(img_ch2_norm, cmap='viridis')
    axes[0, 1].set_title(f'Original Channel {C2_IDX}')
    axes[0, 1].axis('off')

    axes[0, 2].imshow(overlay)
    axes[0, 2].set_title('Overlay (R=Ch1, G=Ch2)')
    axes[0, 2].axis('off')

    # Row 2: full predictions
    axes[1, 0].imshow(pred_ch1, cmap='gray')
    axes[1, 0].set_title('Prediction – Channel 1')
    axes[1, 0].axis('off')

    axes[1, 1].imshow(pred_ch2, cmap='gray')
    axes[1, 1].set_title('Prediction – Channel 2')
    axes[1, 1].axis('off')

    axes[1, 2].imshow(combined_pred)
    axes[1, 2].set_title('Predictions Overlay')
    axes[1, 2].axis('off')

    # Row 3: zoomed patch
    axes[2, 0].imshow(img_ch1_norm[y0:y0+ph, x0:x0+pw], cmap='viridis')
    axes[2, 0].set_title('Zoom – Orig Ch1')
    axes[2, 0].axis('off')

    axes[2, 1].imshow(img_ch2_norm[y0:y0+ph, x0:x0+pw], cmap='viridis')
    axes[2, 1].set_title('Zoom – Orig Ch2')
    axes[2, 1].axis('off')

    axes[2, 2].imshow(combined_pred[y0:y0+ph, x0:x0+pw])
    axes[2, 2].set_title('Zoom – Predictions')
    axes[2, 2].axis('off')

    # --------------------
    # Draw patch box
    # --------------------
    if show_patch_box:
        for ax in axes[0]:
            rect = patches.Rectangle(
                (x0, y0), pw, ph,
                linewidth=2, edgecolor='yellow', facecolor='none'
            )
            ax.add_patch(rect)

    plt.tight_layout()
    plt.show()


Example visualization

In [86]:
import ipywidgets as widgets
from IPython.display import display

def show_prediction(test_idx):
    visualize_dual_predictions(
        original_img=all_slices[test_idx],
        pred_ch1=all_pred_masks_ch1[test_idx],
        pred_ch2=all_pred_masks_ch2[test_idx],
        figsize=(18, 10),
        patch_size=128,
        transpose=True,   # swap axes for correct orientation
        seed=42
)


slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(all_slices) - 1,
    step=1,
    description='Slice:',
    continuous_update=False
)

widgets.interact(show_prediction, test_idx=slider)


interactive(children=(IntSlider(value=0, continuous_update=False, description='Slice:', max=509), Output()), _…

<function __main__.show_prediction(test_idx)>

In [34]:
all_predictions_cleaned = []

for i in tqdm(range(len(all_pred_masks_ch1))):
    # Stack channels: (1, H, W, 2)
    temp = np.expand_dims(
        np.stack(
            [all_pred_masks_ch1[i], all_pred_masks_ch2[i]],
            axis=-1
        ),
        axis=0
    )

    # Step 1: binarize + correct
    predictions, _ = binarize_and_correct_tiff(temp)

    # Step 2: clean cilia channel using red channel
    predictions_cleaned = keep_connected_blobs(
        predictions,
        target_channel=0,
        reference_channel=1
    )

    # Step 3: clean red channel using cilia channel
    predictions_cleaned = keep_connected_blobs(
        predictions_cleaned,
        target_channel=1,
        reference_channel=0
    )

    # Remove batch dimension -> (H, W, 2)
    all_predictions_cleaned.append(predictions_cleaned[0])
all_predictions_cleaned = np.stack(all_predictions_cleaned, axis=0)


 28%|██▊       | 143/510 [00:16<00:41,  8.88it/s]


KeyboardInterrupt: 

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from numpy import test

def show_prediction(test_idx):
    visualize_dual_predictions(
        original_img=all_slices[test_idx],
        pred_ch1=all_predictions_cleaned[test_idx,...,0],
        pred_ch2=all_predictions_cleaned[test_idx,...,1],
        figsize=(18, 10),
        # gamma=0.8   # optional but often helps
    )


slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(all_slices) - 1,
    step=1,
    description='Slice:',
    continuous_update=False
)

widgets.interact(show_prediction, test_idx=slider)


interactive(children=(IntSlider(value=0, continuous_update=False, description='Slice:', max=509), Output()), _…

<function __main__.show_prediction(test_idx)>

## 8. Summary and Statistics

In [ ]:
def print_inference_summary(all_slices, pred_dict_ch1, pred_dict_ch2):
    """Print summary statistics about inference results."""
    print("\n" + "="*60)
    print("INFERENCE SUMMARY")
    print("="*60)
    
    print(f"\nInput Data:")
    print(f"  Total slices processed: {len(all_slices)}")
    print(f"  Slice shape (C, H, W): {all_slices.shape[1:]}")
    
    print(f"\nChannel 1 Predictions:")
    print(f"  Shape: {all_pred_masks_ch1.shape}")
    print(f"  Min value: {all_pred_masks_ch1.min():.4f}")
    print(f"  Max value: {all_pred_masks_ch1.max():.4f}")
    print(f"  Mean value: {all_pred_masks_ch1.mean():.4f}")
    
    print(f"\nChannel 2 Predictions:")
    print(f"  Shape: {all_pred_masks_ch2.shape}")
    print(f"  Min value: {all_pred_masks_ch2.min():.4f}")
    print(f"  Max value: {all_pred_masks_ch2.max():.4f}")
    print(f"  Mean value: {all_pred_masks_ch2.mean():.4f}")
    
    print(f"\nOutput Files:")
    print(f"  Saved location: {PRED_ROOT}")
    print(f"  Number of files: {len(pred_dict_ch1)}")
    print(f"  Format: *_6channel.tif (6 channels: 4 original + 2 predictions)")
    
    print("\n" + "="*60)

In [ ]:
print_inference_summary(all_slices, pred_dict_ch1, pred_dict_ch2)


INFERENCE SUMMARY

Input Data:
  Total slices processed: 510
  Slice shape (C, H, W): (4, 675, 675)

Channel 1 Predictions:
  Shape: (595, 675, 675)
  Min value: 0.0012
  Max value: 1.0000
  Mean value: 0.0096

Channel 2 Predictions:
  Shape: (595, 675, 675)
  Min value: 0.0000
  Max value: 1.0000
  Mean value: 0.0060

Output Files:
  Saved location: /group/jug/aman/Cilia_Datasets/Predictions
  Number of files: 34
  Format: *_6channel.tif (6 channels: 4 original + 2 predictions)



## Optional: Quality Checks

In [ ]:
def validate_6channel_outputs(pred_root):
    """Validate that all 6-channel outputs were saved correctly."""
    print("\nValidating saved 6-channel outputs...")
    
    output_files = list(pred_root.rglob("*_6channel.tif"))
    print(f"Found {len(output_files)} output files")
    
    for i, fpath in enumerate(output_files[:3]):  # Check first 3
        img = tf.imread(fpath)
        print(f"\n  File {i+1}: {fpath.name}")
        print(f"    Shape: {img.shape}")
        print(f"    Dtype: {img.dtype}")
        if img.ndim == 4:
            Z, C, H, W = img.shape
            print(f"    Channels: {C} (expected: 6)")
            for c in range(C):
                ch_data = img[:, c, :, :]
                print(f"      Ch{c}: min={ch_data.min()}, max={ch_data.max()}, mean={ch_data.mean():.2f}")

In [ ]:
validate_6channel_outputs(PRED_ROOT)


Validating saved 6-channel outputs...
Found 34 output files

  File 1: IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_1_original cropped_6channel.tif
    Shape: (16, 6, 675, 675)
    Dtype: float32
    Channels: 6 (expected: 6)
      Ch0: min=0.0, max=425.0, mean=27.14
      Ch1: min=0.0, max=1902.0, mean=36.66
      Ch2: min=0.0, max=763.0, mean=14.79
      Ch3: min=0.0, max=3849.0, mean=70.20
      Ch4: min=0.0018086343770846725, max=1.0, mean=0.01
      Ch5: min=8.5553929238813e-06, max=0.9999995231628418, mean=0.01

  File 2: IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_3_original cropped_6channel.tif
    Shape: (15, 6, 675, 675)
    Dtype: float32
    Channels: 6 (expected: 6)
      Ch0: min=0.0, max=586.0, mean=37.13
      Ch1: min=0.0, max=2820.0, mean=33.21
      Ch2: min=0.0, max=786.0, mean=13.86
      Ch3: min=0.0, max=5961.0, mean=49.90
      Ch4: min=0.0015139615861698985, max=1.0, mean=0.01
      Ch5: min=2.023554225161206e-05, max=0.9999992847442627, mean=0.01

  File 3: IM

In [ ]:
print("\n✓ Pipeline complete!")


✓ Pipeline complete!


In [43]:
original_image = tf.imread("/group/jug/aman/Cilia_Datasets/extracted/W19 - 2025_Pkhd1_cells/W19 - 2025_Pkhd1 cells/ZDHHC5, Y-tub, Arl13, DAPI/Pkhd1 KO/Cropped original image/IMCD3 Pkhd KO - Zdhhc5, ARL13B, Y-TUB, DAPI_9_original cropped.tif")

In [44]:
print(original_img.shape, original_img.dtype, original_img.min(), original_img.max())


(15, 4, 675, 675) uint16 0 5961
